In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from statsmodels.tsa.deterministic import DeterministicProcess
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

# 1. Load Data
def load_data():
    train = pd.read_csv('/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
    test = pd.read_csv('/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv', parse_dates=['date'])
    stores = pd.read_csv('/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv')
    oil = pd.read_csv('/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv', parse_dates=['date'])
    holidays = pd.read_csv('/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv', parse_dates=['date'])
    transactions = pd.read_csv('/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv', parse_dates=['date'])
    return train, test, stores, oil, holidays, transactions

# 2. Feature Engineering
def feature_engineering(train, test, stores, oil, holidays):
    # Oil prices: interpolation and rolling averages
    oil['date'] = pd.to_datetime(oil['date'])
    oil = oil.set_index('date').resample('D').mean().interpolate(method='linear').reset_index()
    oil['oil_lags_1'] = oil['dcoilwtico'].shift(1)
    oil['oil_rolling_7'] = oil['dcoilwtico'].rolling(7).mean()
    
    # Holidays: simplified
    holidays = holidays[holidays['transferred'] == False]
    holidays = holidays.drop_duplicates(subset=['date'])
    
    # Combine train and test for processing
    df = pd.concat([train, test], axis=0)
    df['date'] = pd.to_datetime(df['date'])
    
    # Merge with stores, oil, and holidays
    df = df.merge(stores, on='store_nbr', how='left')
    df = df.merge(oil, on='date', how='left')
    df = df.merge(holidays[['date', 'type', 'locale']], on='date', how='left')
    
    # Time features
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    
    # Categorical encoding
    le = LabelEncoder()
    for col in ['family', 'city', 'state', 'type_x', 'type_y', 'locale']:
        df[col] = le.fit_transform(df[col].astype(str))
        
    return df

# 3. Boosted Hybrid Model
class BoostedHybrid:
    def __init__(self, model_1, model_2):
        self.model_1 = model_1
        self.model_2 = model_2
        self.y_columns = None

    def fit(self, X_1, X_2, y):
        # Train model_1 (Linear Regression for Trend)
        self.model_1.fit(X_1, y)
        y_fit = pd.DataFrame(self.model_1.predict(X_1), index=X_1.index, columns=y.columns)
        
        # Calculate residuals
        y_resid = y - y_fit
        
        # Train model_2 (LightGBM for Residuals)
        self.model_2.fit(X_2, y_resid)
        self.y_columns = y.columns

    def predict(self, X_1, X_2):
        y_pred_1 = pd.DataFrame(self.model_1.predict(X_1), index=X_1.index, columns=self.y_columns)
        y_pred_2 = pd.DataFrame(self.model_2.predict(X_2), index=X_2.index, columns=self.y_columns)
        return y_pred_1 + y_pred_2

# 4. Main Execution
def main():
    train, test, stores, oil, holidays, transactions = load_data()
    
    # Target transformation
    train['sales'] = np.log1p(train['sales'])
    
    # Filter training data to recent years for efficiency and relevance
    train = train[train['date'] >= '2017-01-01']
    
    df = feature_engineering(train, test, stores, oil, holidays)
    
    # Prepare X and y
    y = train.set_index(['store_nbr', 'family', 'date'])['sales'].unstack(['store_nbr', 'family']).fillna(0)
    
    # Deterministic Process for Trend
    dp = DeterministicProcess(
        index=y.index,
        constant=True,
        order=1,
        drop=True,
    )
    X_1 = dp.in_sample()
    X_test_1 = dp.out_of_sample(steps=16)
    
    # Features for Model 2 (LightGBM)
    # We use features that are available at prediction time
    # (No short lags unless recursive, but we'll use time and store features)
    features = ['store_nbr', 'family', 'onpromotion', 'dayofweek', 'month', 'year', 'day', 'is_weekend', 'oil_lags_1', 'oil_rolling_7', 'city', 'state', 'type_x', 'type_y', 'locale']
    
    X_2 = df[df['date'] < '2017-08-16'][features]
    X_test_2 = df[df['date'] >= '2017-08-16'][features]
    
    # Since we need to match the multi-output shape of y, we'll train one model per (store, family) 
    # OR we use a single model with store and family as features. 
    # For a top score, we need to handle the residuals carefully.
    
    # Here's a simplified high-performance approach:
    # 1. Trend model on total sales or grouped sales
    # 2. Residual model on the rest
    
    # To keep it "copy-paste and run", I'll provide a unified LightGBM approach with the best features.
    
    print("Training LightGBM model...")
    # Add more features: Lags (safe lags >= 16)
    for lag in [16, 17, 18, 19, 20, 21, 22, 28, 30]:
        df[f'lag_{lag}'] = df.groupby(['store_nbr', 'family'])['sales'].shift(lag)
        
    # Re-split
    train_df = df[df['date'] < '2017-08-16'].dropna(subset=['lag_28'])
    test_df = df[df['date'] >= '2017-08-16']
    
    features = features + [f'lag_{lag}' for lag in [16, 17, 18, 19, 20, 21, 22, 28, 30]]
    
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
    }
    
    # Multi-seed ensemble
    final_preds = np.zeros(len(test_df))
    seeds = [42, 123, 2024]
    
    for seed in seeds:
        params['random_state'] = seed
        model = lgb.LGBMRegressor(**params)
        model.fit(train_df[features], train_df['sales'])
        final_preds += np.expm1(model.predict(test_df[features])) / len(seeds)
        
    # Submission
    submission = pd.DataFrame({'id': test_df['id'], 'sales': final_preds})
    submission['sales'] = submission['sales'].clip(0, None)
    submission.to_csv('submission.csv', index=False)
    print("Submission saved to submission.csv")

if __name__ == "__main__":
    main()


Training LightGBM model...
Submission saved to submission.csv
